<a href="https://colab.research.google.com/github/lazarosgogos/ML-exercises/blob/main/ML_Project_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# -- DATA ENGINEERING --

## Load libraries

In [26]:
import pandas as pd
import numpy as np
!pip --quiet install ydata-profiling
from ydata_profiling import ProfileReport
from scipy.stats import zscore

## Fetch dataset

In [2]:
!gdown 1554FRLDBIb04_hAElnVVpX_Vjdtar5cz -O bankloan.csv

Downloading...
From: https://drive.google.com/uc?id=1554FRLDBIb04_hAElnVVpX_Vjdtar5cz
To: /content/bankloan.csv
100% 63.3M/63.3M [00:00<00:00, 84.9MB/s]


In [3]:
df = pd.read_csv('bankloan.csv')
df.describe()

,Row ID,id,member_id,loan_amnt,funded_amnt,int_rate,installment,annual_inc,dti,delinq_2yrs,...,mths_since_last_major_derog,annual_inc_joint,dti_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,Unnamed: 50,36months,60months
count,368.000000,2.129990e+05,2.129990e+05,212999.000000,212999.000000,212999.000000,212999.000000,2.129990e+05,212999.000000,212999.000000,...,62365.000000,441.000000,439.000000,212999.000000,212999.000000,2.129990e+05,2.129990e+05,0.0,511.000000,511.000000
mean,184.500000,6.103515e+07,6.515927e+07,15257.965530,15257.965530,12.401658,440.842921,7.780071e+04,19.360817,0.347462,...,45.468356,107574.096327,18.320114,0.005718,261.951652,1.416537e+05,3.444425e+04,NaN,0.622309,0.377691
std,106.376689,4.734904e+06,5.215173e+06,8611.713377,8611.713377,4.249365,245.858646,8.188065e+04,31.925871,0.921209,...,22.645675,47921.057382,7.230012,0.081045,2215.188372,1.568766e+05,3.531827e+04,NaN,0.485285,0.485285
min,1.000000,5.670500e+04,7.082500e+04,1000.000000,1000.000000,5.320000,30.120000,0.000000e+00,0.000000,0.000000,...,0.000000,17950.000000,3.050000,0.000000,0.000000,0.000000e+00,0.000000e+00,NaN,0.000000,0.000000
25%,92.750000,5.783411e+07,6.158651e+07,8500.000000,8500.000000,9.170000,262.230000,4.700000e+04,12.660000,0.000000,...,28.000000,75001.000000,13.185000,0.000000,0.000000,3.143300e+04,1.460000e+04,NaN,0.000000,0.000000
50%,184.500000,6.137900e+07,6.549753e+07,14000.000000,14000.000000,12.290000,382.870000,6.500000e+04,18.720000,0.000000,...,45.000000,100000.000000,17.750000,0.000000,0.000000,8.138600e+04,2.510000e+04,NaN,1.000000,0.000000
75%,276.250000,6.503778e+07,6.956436e+07,20000.000000,20000.000000,14.650000,578.790000,9.250000e+04,25.520000,0.000000,...,63.000000,131000.000000,22.650000,0.000000,0.000000,2.097080e+05,4.260000e+04,NaN,1.000000,1.000000
max,368.000000,6.861687e+07,7.351969e+07,35000.000000,35000.000000,28.990000,1445.460000,9.000000e+06,9999.000000,30.000000,...,171.000000,410000.000000,43.860000,5.000000,380757.000000,4.127799e+06,1.641300e+06,NaN,1.000000,1.000000


In [4]:
df.head()

,Row ID,id,member_id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,...,application_type,annual_inc_joint,dti_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,Unnamed: 50,36months,60months
0,1.0,60516983,64537751,20000,20000,36 months,12.29,667.06,C,C1,...,INDIVIDUAL,NaN,NaN,0,0,52303,41000,NaN,1.0,0.0
1,2.0,60187139,64163931,11000,11000,36 months,12.69,369.00,C,C2,...,INDIVIDUAL,NaN,NaN,0,332,175731,13100,NaN,1.0,0.0
2,3.0,60356453,64333218,7000,7000,36 months,9.99,225.84,B,B3,...,INDIVIDUAL,NaN,NaN,0,0,202012,16300,NaN,1.0,0.0
3,4.0,59955769,63900496,10000,10000,36 months,10.99,327.34,B,B4,...,INDIVIDUAL,NaN,NaN,0,0,108235,34750,NaN,1.0,0.0
4,5.0,58703693,62544456,9550,9550,36 months,19.99,354.87,E,E4,...,INDIVIDUAL,NaN,NaN,0,0,45492,14100,NaN,1.0,0.0


In [ ]:
profile = ProfileReport(df, title='Bankloan profiling report')
profile

## Q2.1 - What is the maximum, mean and minimum value of `loan_amnt`?
Based on the ydata-profiler output, the minimum value is 1000, the mean is 15257.965530 and the maximum value is 35000

## Q2.2 - Which columns can be removed from our dataframe?
We can remove all the ID columns, as they provide no useful information for our model. We can remove columns that are full of NaN values. We may remove highly correlated columns as well (e.g. purpose and title). We can also remove columns of which all rows consist of the same value. Finally we can impute some of the missing values.

In [25]:
# remove unnecessary columns
clean_df = df.drop(columns=['Row ID', 'id', 'member_id', 'loan_status',
                            'total_rec_late_fee',
                            'funded_amnt', 'purpose',
                            'recoveries', 'collection_recovery_fee',
                            'collections_12_mths_ex_med', 'annual_inc_joint',
                            'dti_joint', 'acc_now_delinq',
                            'tot_coll_amt', 'Unnamed: 50',
                            '36months', '60months', 'last_pymnt_d',
                            'next_pymnt_d'])
# fill any missing value left with 0
clean_df['emp_title'] = clean_df['emp_title'].replace(np.nan, 'Other')
clean_df['emp_length'] = clean_df['emp_length'].replace(np.nan, '<1 year')
clean_df.replace(np.nan, 0, inplace=True)

In [24]:
pd.set_option('display.max_columns', None)
clean_df

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,purpose,title,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,total_pymnt,total_rec_prncp,total_rec_int,last_pymnt_amnt,last_credit_pull_d,mths_since_last_major_derog,application_type,tot_cur_bal,total_rev_hi_lim
0,20000,36 months,12.29,667.06,C,C1,Accounting Clerk,1 year,OWN,65000.00,Source Verified,15-Sep,debt_consolidation,Debt consolidation,20.72,0,Sep-00,1,0.0,25,0,31578,77.0,42,w,0.00,0.00,0.00,0.00,0.00,16-Jan,0.0,INDIVIDUAL,52303,41000
1,11000,36 months,12.69,369.00,C,C2,Accounts Payable Lead,7 years,MORTGAGE,40000.00,Source Verified,15-Sep,debt_consolidation,Debt consolidation,24.57,0,2-Sep,0,36.0,13,1,5084,38.8,41,w,0.00,10043.49,9942.67,100.81,10059.00,16-Jan,79.0,INDIVIDUAL,175731,13100
2,7000,36 months,9.99,225.84,B,B3,Nurse,6 years,MORTGAGE,32000.00,Source Verified,15-Sep,debt_consolidation,Debt consolidation,32.41,0,6-Feb,1,0.0,18,0,12070,74.0,36,f,0.00,221.96,167.56,54.40,225.84,16-Jan,0.0,INDIVIDUAL,202012,16300
3,10000,36 months,10.99,327.34,B,B4,Service Manager,10+ years,MORTGAGE,48000.00,Source Verified,15-Sep,credit_card,Credit card refinancing,30.98,0,Oct-99,2,0.0,18,0,22950,66.0,41,f,0.00,315.13,235.76,79.37,327.34,16-Jan,0.0,INDIVIDUAL,108235,34750
4,9550,36 months,19.99,354.87,E,E4,Other,<1 year,RENT,32376.00,Verified,15-Sep,debt_consolidation,Debt consolidation,32.54,0,Nov-99,3,69.0,9,0,4172,29.6,26,w,0.00,333.66,195.78,137.88,354.87,16-Jan,69.0,INDIVIDUAL,45492,14100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212994,20000,36 months,13.33,677.07,C,C3,Vp sale,5 years,RENT,120000.00,Source Verified,15-Jul,debt_consolidation,Debt consolidation,9.04,0,2-Jul,0,45.0,6,0,7,0.1,34,w,17147.54,4120.02,2852.46,1267.56,700.00,16-Jan,45.0,INDIVIDUAL,21474,5400
212995,6000,36 months,11.53,197.95,B,B5,Owner,10+ years,RENT,25000.00,Not Verified,15-Jul,credit_card,Credit card refinancing,2.21,0,5-Sep,2,0.0,3,1,2176,52.0,11,w,5284.89,1010.88,715.11,295.77,197.95,16-Jan,60.0,INDIVIDUAL,2176,4200
212996,18000,60 months,19.19,468.82,E,E3,Production Processor,< 1 year,RENT,120000.00,Source Verified,15-Jul,debt_consolidation,Debt consolidation,7.76,0,9-Feb,1,30.0,4,0,3229,21.1,5,w,16869.83,2793.73,1130.17,1663.56,468.82,16-Jan,0.0,INDIVIDUAL,8414,15300
212997,7050,36 months,15.61,246.51,D,D1,Other,<1 year,MORTGAGE,18614.27,Verified,15-Jul,other,Other,21.53,1,Mar-85,0,5.0,12,0,10256,38.3,25,w,6090.46,1472.95,959.54,513.41,246.51,16-Jan,0.0,INDIVIDUAL,159660,26800


## Q2.3 - Perform other preprocessing methods of your choice
We will perform z-score normalization on numerical columns. We could also convert columns with categorical values to encoded integers.

In [28]:
numerical_cols = clean_df.select_dtypes(include=[np.number]).columns
zscore_df = clean_df.copy()
zscore_df[numerical_cols] = zscore_df[numerical_cols].apply(zscore)
zscore_df

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,title,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,total_pymnt,total_rec_prncp,total_rec_int,last_pymnt_amnt,last_credit_pull_d,mths_since_last_major_derog,application_type,tot_cur_bal,total_rev_hi_lim
0,0.550651,36 months,-0.026276,0.920112,C,C1,Accounting Clerk,1 year,OWN,-0.156334,Source Verified,15-Sep,Debt consolidation,0.042573,-0.377181,Sep-00,0.505032,-0.755781,2.299798,-0.353802,0.552495,0.989781,1.386383,w,-1.698819,-1.047781,-0.894154,-0.975533,-0.967782,16-Jan,-0.553632,INDIVIDUAL,-0.569562,0.185620
1,-0.494440,36 months,0.067856,-0.292213,C,C2,Accounts Payable Lead,7 years,MORTGAGE,-0.461657,Source Verified,15-Sep,Debt consolidation,0.163165,-0.377181,2-Sep,-0.655429,0.791311,0.181001,1.155774,-0.524662,-0.607093,1.303293,w,-1.698819,6.666861,9.079186,-0.765466,21.351838,16-Jan,2.731673,INDIVIDUAL,0.217224,-0.604342
2,-0.958925,36 months,-0.567535,-0.874500,B,B3,Nurse,6 years,MORTGAGE,-0.559361,Source Verified,15-Sep,Debt consolidation,0.408735,-0.377181,6-Feb,0.505032,-0.755781,1.063833,-0.353802,-0.240635,0.864372,0.887845,f,-1.698819,-0.877288,-0.726077,-0.862174,-0.466672,16-Jan,-0.553632,INDIVIDUAL,0.384751,-0.513737
3,-0.610561,36 months,-0.332205,-0.461660,B,B4,Service Manager,10+ years,MORTGAGE,-0.363954,Source Verified,15-Sep,Credit card refinancing,0.363943,-0.377181,Oct-99,1.665492,-0.755781,1.063833,-0.353802,0.201710,0.529948,1.303293,f,-1.698819,-0.805722,-0.657667,-0.810142,-0.241456,16-Jan,-0.553632,INDIVIDUAL,-0.213026,0.008657
4,-0.662816,36 months,1.785763,-0.349685,E,E4,Other,<1 year,RENT,-0.554769,Verified,15-Sep,Debt consolidation,0.412807,-0.377181,Nov-99,2.825953,2.209479,-0.525264,-0.353802,-0.561741,-0.991680,0.056949,w,-1.698819,-0.791489,-0.697770,-0.688219,-0.180371,16-Jan,2.315812,INDIVIDUAL,-0.612978,-0.576028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212994,0.550651,36 months,0.218467,0.960827,C,C3,Vp sale,5 years,RENT,0.515377,Source Verified,15-Jul,Debt consolidation,-0.323275,-0.377181,2-Jul,-0.655429,1.178084,-1.054964,-0.353802,-0.731076,-2.224868,0.721666,w,0.355287,2.116904,1.967105,1.665800,0.585428,16-Jan,1.317744,INDIVIDUAL,-0.766080,-0.822360
212995,-1.075046,36 months,-0.205127,-0.987940,B,B5,Owner,10+ years,RENT,-0.644851,Not Verified,15-Jul,Credit card refinancing,-0.537209,-0.377181,5-Sep,1.665492,-0.755781,-1.584663,1.155774,-0.642892,-0.055294,-1.189396,w,-1.065741,-0.271300,-0.176838,-0.359209,-0.528556,16-Jan,1.941536,INDIVIDUAL,-0.889094,-0.856336
212996,0.318408,60 months,1.597500,0.113794,E,E3,Production Processor,< 1 year,RENT,0.515377,Source Verified,15-Jul,Debt consolidation,-0.363368,-0.377181,9-Feb,0.505032,0.533462,-1.408097,-0.353802,-0.600080,-1.347005,-1.687933,w,0.322020,1.098149,0.239502,2.490982,0.072469,16-Jan,-0.553632,INDIVIDUAL,-0.849330,-0.542051
212997,-0.953119,36 months,0.755019,-0.790427,D,D1,Other,<1 year,MORTGAGE,-0.722840,Verified,15-Jul,Other,0.067945,0.708351,Mar-85,-0.655429,-0.540907,0.004435,-0.353802,-0.314386,-0.627994,-0.026141,w,-0.969242,0.083627,0.068346,0.094307,-0.420808,16-Jan,-0.553632,INDIVIDUAL,0.114780,-0.216439


## Q2.4 - Create a `target` column

In [34]:
zscore_df.sub_grade.value_counts().sort_index()

,count
sub_grade,
A1,7401
A2,5773
A3,5397
A4,7151
A5,11150
B1,12376
B2,12042
B3,13159
B4,12806


In [42]:
def make_targets(row):
  if row['sub_grade'] in ['A1', 'A2', 'A3','A4', 'A5', 'B1', 'B2']:
    return 1
  return 0
# zscore_df['target'] = np.where(zscore_df['sub_grade'] in ['A1', 'A2', 'A3','A4',
#                                                           'A5', 'B1', 'B2'], 1, 0)
zscore_df['target'] = zscore_df.apply(make_targets,axis=1)
zscore_df[['sub_grade', 'target']]

,sub_grade,target
0,C1,0
1,C2,0
2,B3,0
3,B4,0
4,E4,0
...,...,...
212994,C3,0
212995,B5,0
212996,E3,0
212997,D1,0
